# 02 — Pré-processamento e Baseline (Regressão Logística)

**Objetivo:** transformar os dados brutos + features Pix em algo que um modelo consiga usar (imputar nulos, codificar categóricas, escalonar), dividir os dados de forma realista no tempo, e treinar um primeiro modelo simples (baseline) pra ter um número de referência antes do XGBoost/Random Forest de julho.

Cada etapa tem uma célula de explicação em texto antes do código, pra você acompanhar o raciocínio.

## 1. Carregar dados e adicionar as features Pix

Reaproveita o `data_loader.py` (maio) e o `pix_features.py` (junho) já prontos.

`N_LINHAS` controla quantas transações são lidas. Para **testar** o notebook rápido, use `50_000`. Para a **execução oficial** do entregável de junho, use `None` (as 590.540 transações) — mas atenção: o dataset completo ocupa ~2,2 GB só no DataFrame, e o notebook cria cópias ao dividir treino/val/teste. Em máquina com 8 GB de RAM, feche os outros programas antes de rodar com `None`.

Duas etapas de memória acontecem aqui, e sem elas a execução com o dataset completo não termina em uma máquina de 8 GB:

- **`del dados`** descarta as duas tabelas brutas (`train_transaction` e `train_identity`) assim que a junção e as features estão prontas. Elas somam cerca de 2 GB e não são mais usadas.
- **`reduzir_precisao`** converte as colunas de `float64` para `float32`. Nenhuma coluna do IEEE-CIS precisa de 15 algarismos significativos — `TransactionAmt` não passa de dezenas de milhares. A redução corta ~29% do pico de memória e altera as métricas apenas na quarta casa decimal.

A conversão vem **depois** da engenharia de features, de propósito: `pix_features` calcula janelas móveis que somam valores ao longo de milhares de linhas, e acúmulo é justamente onde a precisão menor pesa.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data_loader import carregar_dados
from src.features.pix_features import criar_features_pix
from src.features.preprocessor import (
    COLUNA_ALVO,
    construir_preprocessador,
    dividir_temporal,
    identificar_colunas,
    montar_pipeline_modelo,
    reduzir_precisao,
)

# None = dataset completo (execução oficial) | 50_000 = teste rápido
N_LINHAS = None

dados = carregar_dados(nrows=N_LINHAS)
df = criar_features_pix(dados[2])   # dados[2] e o DataFrame ja mesclado
del dados                           # solta as duas tabelas brutas: ~2 GB no dataset completo

df = reduzir_precisao(df)           # float64 -> float32 no caminho da modelagem
print('Shape final (com features Pix):', df.shape)

19:59:39 [INFO] Carregando train_transaction.csv...


20:13:03 [INFO] Carregando train_identity.csv...


20:15:35 [INFO] Dados carregados — transações: (590540, 394) | identidade: (144233, 41)


20:15:35 [INFO] Mesclando DataFrames por TransactionID (left join)...


20:16:12 [INFO] DataFrame final: (590540, 434)


...

20:28:03 [INFO] Calculando dispositivo_raro_proxy...


20:30:02 [INFO] Calculando posicao_ciclo_diario_relativa...


20:33:32 [INFO] Features Pix adicionadas: (590540, 440)


20:38:16 [INFO] Precisão reduzida — 2664 MB para 1695 MB (64% do original)


Shape final (com features Pix): (590540, 440)


## 2. Identificar grupos de colunas

Antes de processar, precisamos saber quais colunas são numéricas (viram escala) e quais são categóricas (viram codificação). A função `identificar_colunas` faz isso automaticamente: separa por tipo de dado, já excluindo `TransactionID`/`TransactionDT` (não são features) e `card4`/`card6` (decisão registrada — sem equivalente conceitual em Pix).

In [2]:
numericas, categoricas_baixa, categoricas_alta = identificar_colunas(df)
print('Exemplo de numéricas:', numericas[:5], '...')
print('Categóricas baixa cardinalidade:', categoricas_baixa)
print('Categóricas alta cardinalidade:', categoricas_alta)

20:39:07 [INFO] Colunas identificadas — numéricas: 406 | categóricas baixa cardinalidade: 11 | categóricas alta cardinalidade: 18


Exemplo de numéricas: ['TransactionAmt', 'card1', 'card2', 'card3', 'card5'] ...
Categóricas baixa cardinalidade: ['ProductCD', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'DeviceType']
Categóricas alta cardinalidade: ['P_emaildomain', 'R_emaildomain', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceInfo']


## 3. Split temporal (não aleatório)

Em vez de embaralhar as linhas e separar 70/15/15 aleatoriamente, cortamos por tempo: as transações **mais antigas** viram treino, as **mais recentes** viram teste. Isso é mais realista pra fraude — na vida real você só tem o passado pra prever o futuro, nunca o contrário. Um split aleatório deixaria o modelo "ver" padrões de datas futuras durante o treino, o que infla artificialmente o desempenho.

In [3]:
treino, val, teste = dividir_temporal(df, frac_treino=0.7, frac_val=0.15)

X_treino, y_treino = treino.drop(columns=[COLUNA_ALVO]), treino[COLUNA_ALVO]
X_val, y_val = val.drop(columns=[COLUNA_ALVO]), val[COLUNA_ALVO]
X_teste, y_teste = teste.drop(columns=[COLUNA_ALVO]), teste[COLUNA_ALVO]

print('Taxa de fraude — treino: %.2f%% | val: %.2f%% | teste: %.2f%%' % (
    100 * y_treino.mean(), 100 * y_val.mean(), 100 * y_teste.mean()
))

20:39:27 [INFO] Split temporal — treino: 413378 (70.0%) | val: 88581 (15.0%) | teste: 88581 (15.0%)


Taxa de fraude — treino: 3.52% | val: 3.43% | teste: 3.48%


## 4. Pré-processamento: imputação, codificação e escalonamento

- **Numéricas** → preenche nulo com a **mediana** do treino, depois `StandardScaler` (deixa tudo na mesma escala — importante pra Regressão Logística, que é sensível a magnitude).
- **Categóricas de poucos valores** (`ProductCD`, `DeviceType`, `M1`-`M9`) → preenche nulo com a categoria explícita `"ausente"`, depois vira colunas 0/1 (one-hot).
- **Categóricas de muitos valores** (`P_emaildomain`, `DeviceInfo`, `R_emaildomain`...) → preenche nulo com `"ausente"`, depois vira um número (a frequência daquela categoria no treino) — evita criar centenas de colunas one-hot.

Tudo isso é **ajustado (fit) só com o treino** — validação e teste só usam o que já foi aprendido, nunca ensinam nada ao pré-processador. Essa célula só monta o objeto; ele é usado de verdade dentro do pipeline do modelo, na próxima seção.

> `card1`–`card3` e `card5` são numéricas mascaradas, então caem no grupo das **numéricas** pela regra de dtype de `identificar_colunas` — não neste grupo. `card4` (bandeira) e `card6` (crédito/débito) foram descartadas por decisão registrada: são conceitos exclusivos de cartão, sem equivalente no Pix.

In [4]:
preprocessador = construir_preprocessador(numericas, categoricas_baixa, categoricas_alta)
preprocessador

,transformers,"[('numericas', ...), ('categoricas_baixa', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Verificação: o pré-processador faz mesmo o que a tarefa pede?

Esta célula é o **registro de conclusão da tarefa `m2_p1_2`** (imputação de nulos, encoding de categóricas, escalonamento). Ela roda o pré-processador isoladamente numa amostra do treino, só para inspecionar a saída — o ajuste que vale para o modelo acontece dentro do pipeline, na próxima seção.

Três coisas são conferidas:

1. **Imputação** — nenhum nulo sobra na saída, nem no treino nem na validação;
2. **Escalonamento** — as colunas numéricas saem com média ≈ 0 e desvio ≈ 1;
3. **Encoding** — cada coluna da saída carrega o nome real da coluna de origem.

O item 3 não é detalhe cosmético: em julho, o SHAP (`m3_p1_4`) precisa dizer *qual variável* pesou na decisão. Se o codificador de frequência perdesse o nome da coluna, o gráfico do SHAP mostraria `categoricas_alta__7` em vez de `DeviceInfo`, e a Camada 2 da arquitetura aprovada perderia justamente o que ela existe para entregar.

In [5]:
# Amostra só para inspeção — evita duplicar o dataset inteiro na memória.
AMOSTRA_VERIFICACAO = 50_000

pre_verificacao = construir_preprocessador(numericas, categoricas_baixa, categoricas_alta)
X_amostra = X_treino.head(AMOSTRA_VERIFICACAO)

proc_treino = pre_verificacao.fit_transform(X_amostra)          # fit SÓ com treino
proc_val = pre_verificacao.transform(X_val.head(AMOSTRA_VERIFICACAO))  # val só aplica o aprendido
nomes = pre_verificacao.get_feature_names_out()

print('Colunas de entrada:', X_amostra.shape[1], '-> colunas de saída:', len(nomes))
print('1) Nulos restantes  — treino:', int(np.isnan(proc_treino).sum()),
      '| validação:', int(np.isnan(proc_val).sum()))

bloco_numerico = proc_treino[:, :len(numericas)]
print('2) Numéricas no treino — média: %.3f | desvio: %.3f  (esperado ~0 e ~1)' % (
    bloco_numerico.mean(), bloco_numerico.std()))

print('3) Nomes preservados na saída:')
for nome in list(nomes[:2]) + [n for n in nomes if n.startswith('categoricas_alta__')][:3]:
    print('   ', nome)

del proc_treino, proc_val, X_amostra  # libera a memória da inspeção

Colunas de entrada: 439 -> colunas de saída: 459
1) Nulos restantes  — treino: 0 | validação: 0


2) Numéricas no treino — média: -0.000 | desvio: 0.999  (esperado ~0 e ~1)
3) Nomes preservados na saída:
    numericas__TransactionAmt
    numericas__card1
    categoricas_alta__P_emaildomain
    categoricas_alta__R_emaildomain
    categoricas_alta__id_12


## 5. Baseline com peso de classe (`class_weight='balanced'`)

Com só 3,5% de fraude, um modelo "preguiçoso" que sempre chuta "não é fraude" já acerta 96,5% — inútil. `class_weight='balanced'` faz o modelo **penalizar mais** quando erra a classe rara (fraude), sem duplicar nenhuma linha de dado.

A função `avaliar` mede, além das quatro métricas pedidas, mais três que tornam o resultado interpretável sozinho:

- **precisão** — sem ela o F1 não se lê, porque F1 é a média harmônica de precisão e recall;
- **taxa base** (proporção de fraude no conjunto) — a AUC-PR de um classificador aleatório é igual à taxa base, então sem esse número a AUC-PR não tem régua;
- **`auc_pr_relativo`** — quantas vezes a AUC-PR supera esse acaso.

O limiar de decisão é um parâmetro explícito (`limiar=0.5`), e não mais o padrão implícito do `predict`.

In [6]:
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def avaliar(pipeline, X, y, nome, limiar=0.5):
    probas = pipeline.predict_proba(X)[:, 1]
    # predict() e exatamente proba >= 0.5 na regressao logistica; derivar daqui
    # evita transformar o conjunto inteiro uma segunda vez, e deixa o limiar explicito.
    preds = probas >= limiar

    taxa_base = float(y.mean())
    metricas = {
        'auc_roc': roc_auc_score(y, probas),
        'auc_pr': average_precision_score(y, probas),
        'f1': f1_score(y, preds),
        'recall': recall_score(y, preds),
        'precisao': precision_score(y, preds, zero_division=0),
        'taxa_base': taxa_base,
        'n': int(len(y)),
        'n_fraudes': int(y.sum()),
    }
    # Quanto a AUC-PR supera um classificador aleatorio, que vale a propria taxa base.
    metricas['auc_pr_relativo'] = metricas['auc_pr'] / taxa_base

    print(f"--- {nome} ---")
    print(f"  {metricas['n']} transacoes | {metricas['n_fraudes']} fraudes | taxa base {100*taxa_base:.3f}%")
    for k in ('auc_roc', 'auc_pr', 'f1', 'recall', 'precisao'):
        print(f'  {k}: {metricas[k]:.4f}')
    print(f"  auc_pr_relativo: {metricas['auc_pr_relativo']:.1f}x o acaso")
    return metricas


pipeline_peso = montar_pipeline_modelo(preprocessador, class_weight='balanced')
pipeline_peso.fit(X_treino, y_treino)
metricas_peso = avaliar(pipeline_peso, X_val, y_val, 'class_weight=balanced (validação)')

--- class_weight=balanced (validação) ---
  88581 transacoes | 3042 fraudes | taxa base 3.434%
  auc_roc: 0.8414
  auc_pr: 0.3934
  f1: 0.2248
  recall: 0.6785
  precisao: 0.1347
  auc_pr_relativo: 11.5x o acaso


## 6. Baseline com SMOTE

SMOTE cria exemplos sintéticos de fraude (interpolando entre casos reais de fraude do treino) até equilibrar as classes, antes de treinar. É uma estratégia diferente de `class_weight` — por isso comparamos as duas em pipelines separados, nunca misturadas (decisão já registrada nas anotações de metodologia). O SMOTE só é aplicado dentro do treino — nunca em validação/teste, senão estaríamos avaliando em dado inventado.

In [7]:
pipeline_smote = montar_pipeline_modelo(preprocessador, usar_smote=True)
pipeline_smote.fit(X_treino, y_treino)
metricas_smote = avaliar(pipeline_smote, X_val, y_val, 'SMOTE (validação)')

--- SMOTE (validação) ---
  88581 transacoes | 3042 fraudes | taxa base 3.434%
  auc_roc: 0.8392
  auc_pr: 0.3967
  f1: 0.2206
  recall: 0.6831
  precisao: 0.1316
  auc_pr_relativo: 11.6x o acaso


## 7. Comparação

Com desbalanceamento forte, **AUC-PR** é a métrica mais confiável pra comparar (AUC-ROC pode parecer "boa" mesmo quando o modelo erra bastante as fraudes, exatamente por causa dos 96,5% de exemplos fáceis).

In [8]:
comparacao = pd.DataFrame({'class_weight': metricas_peso, 'smote': metricas_smote}).T
comparacao

,auc_roc,auc_pr,f1,recall,precisao,taxa_base,n,n_fraudes,auc_pr_relativo
class_weight,0.84140,0.393402,0.224824,0.678501,0.134735,0.034341,88581.0,3042.0,11.455591
smote,0.83924,0.396681,0.220630,0.683103,0.131561,0.034341,88581.0,3042.0,11.551089


## 8. Avaliação final no teste

Só agora — depois de já ter escolhido a estratégia melhor na validação — testamos no conjunto de teste (as transações mais recentes, nunca vistas). Isso simula "desempenho real": o número aqui é o mais próximo do que aconteceria em produção.

Ajuste a variável `melhor_pipeline` abaixo pra apontar pro pipeline com melhor AUC-PR na tabela acima antes de rodar esta célula.

In [9]:
melhor_pipeline = pipeline_peso if metricas_peso['auc_pr'] >= metricas_smote['auc_pr'] else pipeline_smote
nome_melhor = 'class_weight=balanced' if melhor_pipeline is pipeline_peso else 'SMOTE'
print('Estratégia escolhida (maior AUC-PR na validação):', nome_melhor)

metricas_teste = avaliar(melhor_pipeline, X_teste, y_teste, f'{nome_melhor} (TESTE — avaliação final)')

Estratégia escolhida (maior AUC-PR na validação): SMOTE


--- SMOTE (TESTE — avaliação final) ---
  88581 transacoes | 3083 fraudes | taxa base 3.480%
  auc_roc: 0.8234
  auc_pr: 0.1840
  f1: 0.2071
  recall: 0.7071
  precisao: 0.1213
  auc_pr_relativo: 5.3x o acaso


## 9. Resultados e próximos passos

### O que este baseline entrega

Com as 590.540 transações, a regressão logística chega a uma AUC-PR de **0,3934** (ponderação de classe) e **0,3967** (SMOTE) na validação, contra uma taxa base de 3,434% — ou seja, cerca de **11,5× melhor que o acaso**. As duas estratégias empatam: a diferença de 0,0033 é menor que a incerteza para 3.042 fraudes, e o sinal da diferença inclusive se inverte em relação à amostra de 50.000 linhas, onde a ponderação vencia. O SMOTE custou várias vezes mais tempo de treino para chegar ao mesmo lugar.

### O achado principal: o desempenho cai no período mais recente

A AUC-PR passa de **0,3967 na validação para 0,1840 no teste** — menos da metade — enquanto:

- as taxas base são praticamente iguais (3,434% e 3,480%), então não é efeito de desbalanceamento;
- a AUC-ROC quase não se move (0,8392 para 0,8234), então o modelo continua ordenando;
- o recall até sobe (0,6831 para 0,7071).

O que se degrada é a **pureza das previsões de maior confiança** no período mais recente: os padrões aprendidos no passado envelhecem. Esse resultado só é visível por causa do corte temporal — um split aleatório teria misturado os períodos e reportado ~0,39 como desempenho do modelo, errando por um fator de dois.

Em termos operacionais no teste: das 3.083 fraudes o modelo recupera cerca de 2.180, marcando aproximadamente 18.000 das 88.581 transações como suspeitas. São 20% de todas as transações, com 8 de cada 10 acusações sendo alarme falso.

### O que fica pendente

- **O limiar de decisão continua em 0,5**, que é o padrão e não foi escolhido. Ajustá-lo na validação — nunca no teste — é a melhoria mais barata disponível, e move precisão e recall em direções opostas conforme a prioridade do problema.
- **A regra de escolha entre as duas estratégias precisa ser fixada por escrito.** A regra automática usada aqui seleciona por maior AUC-PR na validação, o que aponta o SMOTE; os critérios secundários registrados nas anotações de metodologia apontam a ponderação de classe. Escolher olhando o resultado do teste contaminaria o conjunto de teste.
- **O SMOTE interpola colunas já codificadas em one-hot**, gerando linhas com categorias fracionárias que não existem no domínio. Limitação assumida nesta etapa, não corrigida.
- **A execução não é reprodutível bit a bit.** Rodando este mesmo código com a mesma semente em um script separado, as métricas saíram diferentes a partir da terceira casa decimal (AUC-PR de teste 0,1863 contra 0,1840 aqui). A causa é a combinação de `float32` com as operações matriciais paralelas do BLAS: a ordem das somas varia entre execuções e o resultado muda nos últimos dígitos. As conclusões não dependem dessa casa decimal, mas números citados na monografia devem vir de uma execução identificada, não de duas misturadas.

### Sobre o custo de execução

Esta execução levou **1h25**, contra 49 minutos do mesmo experimento rodado como script isolado. A diferença não está no código: com o kernel do Jupyter e o restante do ambiente na memória, a máquina passou a paginar para o disco. A engenharia de features sozinha levou 17 minutos aqui contra cerca de 40 segundos no script. Qualquer estimativa de tempo citada na monografia precisa dizer em que condições foi medida.

### O que julho precisa superar

Estes números são o piso de comparação. XGBoost (`m3_p1_1`) e Random Forest (`m3_p1_2`) precisam melhorar a **precisão sem perder recall** para justificar a complexidade adicional — e devem ser avaliados no mesmo corte temporal, para que a queda entre validação e teste seja comparável. O SHAP (`m3_p1_4`) então explica quais variáveis pesaram nessas decisões.